Tools

In [1]:
from openai.types import model
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def get_weather(location:str) -> str:
    """Get the weather at a location"""
    return f"It's raining in {location}"

@tool
def get_location() -> str:
    """Get User Location"""
    return "Shen Zhen"

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=os.getenv("GOOGLE_API_KEY"),
)
agent = create_agent(
    model=llm
)

model_with_tools = llm.bind_tools([get_weather,get_location])   # 注意是 llm 不是 model
response = model_with_tools.invoke("我现在这里的天气怎么样？")
print(response.tool_calls) #  # 这里只会拿到模型想调用的工具及参数，工具还没真正执行


# 这种就算写2个也是调用一个 可以试一下agent
# agent = create_agent(model=llm, tools=[get_weather, get_location])
# result = agent.invoke({"messages": [("user", "我现在这里的天气怎么样？")]})
# print(result["messages"][-1].content)
# 您所在的深圳目前正在下雨。

[{'name': 'get_location', 'args': {}, 'id': 'c0000fad-848c-4f0d-add8-92ff64821d15', 'type': 'tool_call'}]


## Tool Execution Loops


In [2]:
# 生成 tool calls
messages = [{"role": "user", "content": "深圳今天天气怎么样？"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# 调用工具
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# 生成结果
finally_resp = model_with_tools.invoke(messages)
print(finally_resp)

content='深圳今天下雨。' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f162d-06ab-7901-abb3-5622e7775777-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 104, 'output_tokens': 5, 'total_tokens': 109, 'input_token_details': {'cache_read': 0}}


In [3]:
messages

[{'role': 'user', 'content': '深圳今天天气怎么样？'},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "\\u6df1\\u5733"}'}, '__gemini_function_call_thought_signatures__': {'5df49cec-ff7f-42c2-aaf2-95a84fa4e5fd': 'CugCAQw51sfvRfCGrVhmmzaobtBqR5jsDYmxjHADNCeOIpGhKKVkx7OUcR/DT/MwVe9n4exCAPjdD2hQOETUTA3SYOM3rZDaKxOy+p2j/jvAb42mhs/4FsE2kGIRljIUj0ZmmIfuNoLHS+Cjk7Qk/a5AzjFewGS+fJQaVe4zL8EjbKb11avOVxlmwjwOWzVz5Oc3GSgIqx5G14pZo6zxtFJzcmCctqkY84GG9qBigUc2QEy8GX4rIYdgIsF7i4wbzwJxDn/3bjfSfGY6xJhbzMOQuncCp5qagtPqzMpWLX3oYc+HaFujmuvNhYWRjdk8twWvWGBEx7RFT0MIILiuwE0oaPnDkraWa0A3Zz7Z4ZeARpsUbiqk3pyHfWge6rEhPOKn5UmKaQZuQvaFdmyXyIX+gRrIy6n+BsWWG1dQ9JRH2muGWTzn+yTr5tohb0qXWpwS/Ot4n2QqngSPVg1NFcuCpJ15YkWEl5Kh'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f162d-018e-7511-9275-fb66c9742e13-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '深